<a href="https://colab.research.google.com/github/andreasangi/Prog-Machine-Learning/blob/main/patchcore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile resnet.py
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class ResNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.rete = resnet50(weights = ResNet50_Weights.DEFAULT)      #crea la rete
    self.features = {}
    for par in self.rete.parameters():
      par.requires_grad = False           #blocca l'aggiornamneto dei pesi
    self.rete.layer2.register_forward_hook(self.getFeatures('layer2'))
    self.rete.layer3.register_forward_hook(self.getFeatures('layer3'))
    self.rete.eval();                         #elimina il drop out e altre tecniche di overfitting

  def getFeatures(self, layerName):
    def extractor(mod, input, output):
      self.features[layerName] = output  #per avere informazioni sul layer
    return extractor

  def forward(self, x):
    _ = self.rete(x)          #scartatiamo l'out finale
    return self.features     #restituisce le features esatte

Overwriting resnet.py


In [ ]:

%%writefile patchcore.py
import torch
import torch.nn.functional as F
from sklearn.random_projection import SparseRandomProjection

class PatchCore:
  def __init__(self, fe, d):
    self.extractor = fe
    self.device = d
    self.memory_bank = None

  def fit(self, dl):
    fea_l2 = []                      #crea le liste vuote per le features
    fea_l3 = []
    self.extractor.eval();           #evita di aggiornare i pesi
    with torch.no_grad():
      for images in dl:
        images = images.to(self.device)
        out = self.extractor(images)
        fea_l2.append(out['layer2'].cpu())
        fea_l3.append(out['layer3'].cpu())

      fea_l2 = torch.cat(fea_l2, dim=0)
      fea_l3 = torch.cat(fea_l3, dim=0)       #creiamo due grandi tensori lungo la prima dim
      fea_l3NEW = F.interpolate(fea_l3, size=(fea_l2.shape[2], fea_l2.shape[3]), mode='bilinear', align_corners=False)    #il lay3 è piccolo, va ridimensionacome come il 2
      pc_fea = torch.cat([fea_l2, fea_l3NEW], dim=1) #incolliamo i tensori lungo la dimensione die canali
      B, C, H, W = pc_fea.shape
      self.memory_bank = pc_fea.permute(0, 2, 3, 1).reshape(-1, C)
    print(f"{self.memory_bank.shape[0]} vettori (patch).")

  def subsampling(self, perc):
    mapper = SparseRandomProjection(n_components=128, random_state=42)        #il mapper schiaccia i tenssori mantendno le distanze invariate, 128 canali
    bank_np = self.memory_bank.numpy()                                          #converto il mapper in numpy
    bank_sparse = mapper.fit_transform(self.memory_bank.numpy())     #sparsifica i tensori
    bank_sparse = torch.tensor(bank_sparse)
    num_target = int(self.memory_bank.shape[0] * perc)
    indices = torch.randperm(self.memory_bank.shape[0])[:num_target]        #estraggo gli indici dei vettori in mada da avere un dataset significatico
    self.memory_bank = self.memory_bank[indices]
    print(f"{self.memory_bank.shape[0]} nuova dimensione")

  def predict(self, image_tensor):
    self.extractor.eval()
    with torch.no_grad():
      image_tensor = image_tensor.to(self.device)
      out = self.extractor(image_tensor)        #estraggo patch
      f2 = out['layer2'].cpu()
      f3 = out['layer3'].cpu()
      f3NEW = F.interpolate(f3, size=(f2.shape[2], f2.shape[3]), mode='bilinear', align_corners=False)    #il lay3 è piccolo, va ridimensionacome come il 2
      pc_fea = torch.cat([f2, f3NEW], dim=1)    #creiamo due grandi tensori lungo la prima dim
      B, C, H, W = pc_fea.shape
      test_patches = pc_fea.permute(0, 2, 3, 1).reshape(-1, C)     #incolliamo i tensori lungo la dimensione die canali
      distanze = torch.cdist(test_patches, self.memory_bank)     #calolo le dist
      min_distanze, _ = torch.min(distanze, dim=1)            #trovo la distanza min (le features più simili)
      score_anomalia = torch.max(min_distanze).item()         #trovo la dist max (stranezza peggiore)
      return score_anomalia

  def get_heatmap(self, image_tensor):
    self.extractor.eval()
    with torch.no_grad():
      image_tensor = image_tensor.to(self.device)
      out = self.extractor(image_tensor)
      f2 = out['layer2'].cpu()
      f3 = out['layer3'].cpu()
      f3NEW = F.interpolate(f3, size=(f2.shape[2], f2.shape[3]), mode='bilinear', align_corners=False)
      pc_fea = torch.cat([f2, f3NEW], dim=1)
      B, C, H, W = pc_fea.shape

      test_patches = pc_fea.permute(0, 2, 3, 1).reshape(-1, C)
      distanze = torch.cdist(test_patches, self.memory_bank)
      min_distanze, _ = torch.min(distanze, dim=1)
      score_anomalia = torch.max(min_distanze).item()


      heatmap = min_distanze.reshape(H, W).numpy()

      return score_anomalia, heatmap

Overwriting patchcore.py


In [ ]:
%%writefile dataset.py
import os
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

pp = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),        #da la forma all'immaigne per darrla alla rete
    transforms.ToTensor(),           #trasforma in un tensore
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class immagini:
  def __init__(self, metal_nut, trasnform = pp):
    self.transform = trasnform
    self.image_paths = []
    for root, dirs, files in os.walk(metal_nut):
      for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
          self.image_paths.append(os.path.join(root, file))
    if len(self.image_paths) == 0:
      print("0")

  def __len__(self):
    return len(self.image_paths)

  def __getitem__(self, idx):
    img_path = self.image_paths[idx]
    image = Image.open(img_path).convert('RGB')
    if self.transform:
      image = self.transform(image)
    return image

Overwriting dataset.py


In [ ]:
%%writefile test.py

import os
import numpy as np
import torch
from PIL import Image
import torchvision.transforms as transforms


class tester:
    def __init__(self, modello, metal_nut_test, threshold):
        self.threshold = threshold
        self.modello = modello
        self.metal_nut_test = metal_nut_test

        self.TP = 0
        self.FP = 0
        self.TN = 0
        self.FN = 0

        self.preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])


    def testa(self):

        # Reset dei contatori ogni volta che viene eseguito il test
        self.TP = 0
        self.FP = 0
        self.TN = 0
        self.FN = 0

        # Ground truth e predizioni
        y_v = []
        y_p = []

        # Score separati per analisi
        good_scores = []
        anomaly_scores = []

        # Score divisi anche per categoria
        category_scores = {}

        # ---------------------------------------------------------
        # TEST DI TUTTE LE IMMAGINI
        # ---------------------------------------------------------

        for cat in sorted(os.listdir(self.metal_nut_test)):

            cat_path = os.path.join(self.metal_nut_test, cat)

            # Ignora eventuali file presenti nella cartella test
            if not os.path.isdir(cat_path):
                continue

            # good = 0, tutte le altre categorie = anomalia
            if cat == 'good':
                anomalia = 0
            else:
                anomalia = 1

            category_scores[cat] = []

            campioni = [
                f for f in os.listdir(cat_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]

            for immagine in campioni:

                img_path = os.path.join(cat_path, immagine)

                # ResNet lavora con immagini RGB
                img = Image.open(img_path).convert("RGB")

                img_tensor = self.preprocess(img).unsqueeze(0)

                # Calcolo anomaly score
                score = self.modello.predict(img_tensor)

                # Converte tensor scalare -> float Python
                if torch.is_tensor(score):
                    score = score.item()
                else:
                    score = float(score)

                # Salvo lo score della categoria
                category_scores[cat].append(score)

                # Salvo separatamente good/anomaly
                if anomalia == 0:
                    good_scores.append(score)
                else:
                    anomaly_scores.append(score)

                # Ground truth
                y_v.append(anomalia)

                # Predizione usando threshold
                if score > self.threshold:
                    y_p.append(1)
                else:
                    y_p.append(0)


        # ---------------------------------------------------------
        # CONFUSION MATRIX
        # ---------------------------------------------------------

        for vero, predetto in zip(y_v, y_p):

            if vero == 1 and predetto == 1:
                self.TP += 1

            elif vero == 0 and predetto == 1:
                self.FP += 1

            elif vero == 0 and predetto == 0:
                self.TN += 1

            elif vero == 1 and predetto == 0:
                self.FN += 1


        # ---------------------------------------------------------
        # METRICHE
        # ---------------------------------------------------------

        totale = self.TP + self.TN + self.FP + self.FN

        accuracy = (
            (self.TP + self.TN) / totale
            if totale > 0
            else 0
        )

        precision = (
            self.TP / (self.TP + self.FP)
            if (self.TP + self.FP) > 0
            else 0
        )

        recall = (
            self.TP / (self.TP + self.FN)
            if (self.TP + self.FN) > 0
            else 0
        )

        f1_score = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )


        confusion_matrix = [
            [self.TN, self.FP],
            [self.FN, self.TP]
        ]


        # ---------------------------------------------------------
        # ANALISI DEGLI ANOMALY SCORE
        # ---------------------------------------------------------

        print("\n--- SCORE ANALYSIS ---")

        if len(good_scores) > 0:
            print("\nGOOD:")
            print(f"Samples: {len(good_scores)}")
            print(f"Min:     {np.min(good_scores):.4f}")
            print(f"Max:     {np.max(good_scores):.4f}")
            print(f"Mean:    {np.mean(good_scores):.4f}")
            print(f"Std:     {np.std(good_scores):.4f}")

        if len(anomaly_scores) > 0:
            print("\nANOMALIES:")
            print(f"Samples: {len(anomaly_scores)}")
            print(f"Min:     {np.min(anomaly_scores):.4f}")
            print(f"Max:     {np.max(anomaly_scores):.4f}")
            print(f"Mean:    {np.mean(anomaly_scores):.4f}")
            print(f"Std:     {np.std(anomaly_scores):.4f}")

        print(f"\nThreshold: {self.threshold:.4f}")


        # ---------------------------------------------------------
        # SCORE PER CATEGORIA
        # ---------------------------------------------------------

        print("\n--- SCORE BY CATEGORY ---")

        for cat, scores in category_scores.items():

            if len(scores) > 0:
                print(
                    f"{cat:12s} | "
                    f"N = {len(scores):3d} | "
                    f"Mean = {np.mean(scores):.4f} | "
                    f"Min = {np.min(scores):.4f} | "
                    f"Max = {np.max(scores):.4f}"
                )


        # ---------------------------------------------------------
        # RISULTATI FINALI
        # ---------------------------------------------------------

        print("\n--- RESULTS ---")

        print(f"Accuracy:  {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1 Score:  {f1_score:.4f}")


        print("\nConfusion Matrix:")
        print("                 Predicted Normal   Predicted Anomaly")
        print(f"Actual Normal          {self.TN:<10}       {self.FP}")
        print(f"Actual Anomaly         {self.FN:<10}       {self.TP}")


        # ---------------------------------------------------------
        # RETURN
        # ---------------------------------------------------------

        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1_score,

            "TP": self.TP,
            "TN": self.TN,
            "FP": self.FP,
            "FN": self.FN,

            "confusion_matrix": confusion_matrix,

            "threshold": self.threshold,

            "good_scores": good_scores,
            "anomaly_scores": anomaly_scores,

            "category_scores": category_scores
        }

Overwriting test.py


In [ ]:
import importlib
import test

importlib.reload(test)

tester = test.tester

In [ ]:
import torch
import numpy as np

from dataset import immagini
from resnet import ResNet
from patchcore import PatchCore
from test import tester

from google.colab import drive
from torch.utils.data import DataLoader, random_split


drive.mount('/content/drive')

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ---------------------------------------------------------
# PATH
# ---------------------------------------------------------

metal_nut = '/content/drive/MyDrive/metal_nut/train/good'
metal_nut_test = '/content/drive/MyDrive/test/test_perspective'


# ---------------------------------------------------------
# DATASET COMPLETO: SOLO GOOD
# ---------------------------------------------------------

dataset = immagini(metal_nut)

print("Numero immagini good:", len(dataset))


# ---------------------------------------------------------
# TRAIN / VALIDATION SPLIT
# ---------------------------------------------------------

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=generator
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))


# ---------------------------------------------------------
# DATALOADER
# ---------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=False
)

# IMPORTANTE:
# batch_size = 1 perché vogliamo uno score per ogni immagine
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False
)


# ---------------------------------------------------------
# RESNET-50
# ---------------------------------------------------------

estrattore = ResNet().to(device)


# ---------------------------------------------------------
# PATCHCORE
# ---------------------------------------------------------

modello_patchcore = PatchCore(
    fe=estrattore,
    d=device
)


# Costruzione memory bank SOLO con training set
modello_patchcore.fit(train_loader)

# Salvataggio del modello finale
torch.save({
    "resnet_state_dict": estrattore.state_dict(),
    "memory_bank": modello_patchcore.memory_bank
}, "/content/patchcore_model.pth")

# Per ora manteniamo lo stesso subsampling
modello_patchcore.subsampling(0.20)


# ---------------------------------------------------------
# CALCOLO SCORE VALIDATION
# ---------------------------------------------------------

validation_scores = []

print("\n--- VALIDATION SCORE CALCULATION ---")

for img_tensor in val_loader:

    score = modello_patchcore.predict(img_tensor)

    if torch.is_tensor(score):
        score = score.item()
    else:
        score = float(score)

    validation_scores.append(score)


validation_scores = np.array(validation_scores)


# ---------------------------------------------------------
# ANALISI VALIDATION
# ---------------------------------------------------------

print("\n--- VALIDATION SCORES ---")

print(f"Samples: {len(validation_scores)}")
print(f"Min:     {validation_scores.min():.4f}")
print(f"Max:     {validation_scores.max():.4f}")
print(f"Mean:    {validation_scores.mean():.4f}")
print(f"Std:     {validation_scores.std():.4f}")


# ---------------------------------------------------------
# THRESHOLD
# ---------------------------------------------------------

threshold = np.percentile(validation_scores, 90)

print("\n--- THRESHOLD ---")
print(f"90th percentile threshold: {threshold:.4f}")


# ---------------------------------------------------------
# TEST FINALE
# ---------------------------------------------------------

tester_model = tester(
    modello_patchcore,
    metal_nut_test,
    threshold
)

risultati = tester_model.testa()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Numero immagini good: 220
Training images: 176
Validation images: 44
137984 vettori (patch).
27596 nuova dimensione

--- VALIDATION SCORE CALCULATION ---

--- VALIDATION SCORES ---
Samples: 44
Min:     30.9223
Max:     40.9732
Mean:    33.3027
Std:     1.8735

--- THRESHOLD ---
90th percentile threshold: 35.6696

--- SCORE ANALYSIS ---

GOOD:
Samples: 20
Min:     30.1225
Max:     42.9033
Mean:    36.1235
Std:     3.3711

ANOMALIES:
Samples: 80
Min:     33.2780
Max:     53.6024
Mean:    42.8634
Std:     3.8560

Threshold: 35.6696

--- SCORE BY CATEGORY ---
bent         | N =  20 | Mean = 44.2157 | Min = 39.4527 | Max = 53.6024
color        | N =  20 | Mean = 42.3865 | Min = 35.0783 | Max = 51.1736
flip         | N =  20 | Mean = 45.2908 | Min = 43.3480 | Max = 48.1519
good         | N =  20 | Mean = 36.1235 | Min = 30.1225 | Max = 42.9033
scratch 

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

#riporta l'immagine una versione con colori visibili
def denormalize(img_tensor, mean, std):
    img = img_tensor.clone().squeeze().cpu().numpy().transpose(1, 2, 0)
    img = np.array(std) * img + np.array(mean)
    return np.clip(img, 0, 1)

dir_out = '/content/drive/MyDrive/machine/MLmigliore/data/heatmap_perspective'
os.makedirs(dir_out, exist_ok=True)
cmap = plt.get_cmap('RdYlGn_r') #carica mappa a colori
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
cont = 1

for dir in os.listdir(metal_nut_test):
  path = os.path.join(metal_nut_test, dir)
  if not os.path.isdir(path) or dir.lower() == 'good':
    continue
  campioni = [f for f in os.listdir(path) if f.lower().endswith(('.png'))]

  for immagine in campioni:
    img_path = os.path.join(path, immagine)
    img = Image.open(img_path).convert('RGB')
    img_tensor = tester_model.preprocess(img).unsqueeze(0).to(device)
    score, heatmap = modello_patchcore.get_heatmap(img_tensor)     #ottiene heatmap dalla patchcore
    heatmap = (heatmap - np.min(heatmap)) / (np.max(heatmap) - np.min(heatmap) + 1e-8)   #normalizza

    img_rgb = denormalize(img_tensor, mean, std)
    heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_color = cmap(heatmap_resized)[:, :, :3]

    #60% nut, 40% heatmap
    overlayed_img = heatmap_color * 0.4 + img_rgb * 0.6
    overlayed_img = np.clip(overlayed_img, 0, 1)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Reale: {path}", color='black', fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(overlayed_img)
    axes[1].set_title(f"Score Anomalia: {score:.2f}", color='red', fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    nome_file = f"{dir}_{cont}.png"
    percorso_completo = os.path.join(dir_out, nome_file)
    plt.savefig(percorso_completo, bbox_inches='tight', dpi=150)
    plt.close()

    cont += 1

print(f"end")

end
